In [ ]:
from IPython.display import HTML, display

_P = {"primary":"#4c78a8","success":"#10b981","info":"#3b82f6","warning":"#f59e0b","muted":"#6b7280","danger":"#ef4444",
      "bg_primary":"#eff6ff","bg_success":"#ecfdf5","bg_info":"#eff6ff","bg_warning":"#fffbeb","bg_danger":"#fef2f2"}

def _stat_card(value, label, sub, kind="primary"):
    c = _P[kind]; bg = _P.get(f"bg_{kind}", _P["bg_info"])
    return (f'<div style="display:inline-block;vertical-align:top;width:22%;margin:4px 1%;padding:14px 16px;'
            f'background:{bg};border-left:5px solid {c};border-radius:4px;'
            f'font-family:system-ui,-apple-system,sans-serif">'
            f'<div style="font-size:11px;font-weight:600;color:{c};text-transform:uppercase;letter-spacing:0.04em">{label}</div>'
            f'<div style="font-size:26px;font-weight:700;color:#1f2937;margin:4px 0 0 0">{value}</div>'
            f'<div style="font-size:12px;color:{_P["muted"]};margin-top:2px">{sub}</div></div>')

def _step(label, sub, kind="primary"):
    c = _P[kind]; bg = _P.get(f"bg_{kind}", _P["bg_info"])
    return (f'<div style="display:inline-block;vertical-align:middle;min-width:138px;padding:10px 12px;'
            f'margin:4px 0;background:{bg};border:2px solid {c};border-radius:6px;text-align:center;'
            f'font-family:system-ui,-apple-system,sans-serif">'
            f'<div style="font-weight:600;color:#1f2937;font-size:13px">{label}</div>'
            f'<div style="color:{_P["muted"]};font-size:11px;margin-top:2px">{sub}</div></div>')

_arrow = f'<span style="display:inline-block;vertical-align:middle;margin:0 4px;color:{_P["muted"]};font-size:20px">&rarr;</span>'

cards = [
    _stat_card('17', 'endpoints', 'FastAPI tour', 'primary'),
    _stat_card('curl', 'examples', 'per endpoint', 'info'),
    _stat_card('live', 'catalog', 'vs app drift audit', 'warning'),
    _stat_card('upload', 'case intake', 'multimodal', 'success')
]
display(HTML('<div style="margin:8px 0">' + ''.join(cards) + '</div>'))

steps = [
    _step('Load catalog', '17 endpoints', 'primary'),
    _step('Show curl', 'examples', 'info'),
    _step('Response shapes', 'documented', 'warning'),
    _step('Drift audit', 'catalog vs app', 'success')
]
display(HTML(
    '<div style="margin:10px 0 4px 0;font-family:system-ui,-apple-system,sans-serif;'
    'font-weight:600;color:#1f2937">API endpoint tour</div>'
    '<div style="margin:6px 0">' + _arrow.join(steps) + '</div>'
))


In [ ]:
# Install the pinned DueCare packages for this notebook.
import glob
import subprocess
import sys

PACKAGES = ['duecare-llm-core==0.1.0', 'duecare-llm-domains==0.1.0']
DUECARE_PACKAGES = [package for package in PACKAGES if package.startswith('duecare-')]
EXTRA_PACKAGES = [package for package in PACKAGES if not package.startswith('duecare-')]

def _pip_install(items):
    if items:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *items])

def _wheel_install():
    wheel_patterns = [
        '/kaggle/input/duecare-llm-wheels/*.whl',
        '/kaggle/input/datasets/taylorsamarel/duecare-llm-wheels/*.whl',
        '/kaggle/input/**/*.whl',
    ]
    wheel_files = []
    for pattern in wheel_patterns:
        wheel_files = sorted(glob.glob(pattern, recursive=True))
        if wheel_files:
            break
    if not wheel_files:
        return False
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', *wheel_files])
    if EXTRA_PACKAGES:
        _pip_install(EXTRA_PACKAGES)
    print(f'Installed {len(wheel_files)} wheel files via attached Kaggle dataset.')
    return True

def _duecare_importable():
    try:
        import importlib
        for mod in ('duecare.core',):
            importlib.invalidate_caches()
            importlib.import_module(mod)
        return True
    except Exception:
        return False

install_source = 'pypi'
try:
    _pip_install(PACKAGES)
except Exception as exc:
    print(f'Pinned PyPI install failed ({exc.__class__.__name__}). Falling back to Kaggle wheels for DueCare packages.')
    if not _wheel_install() and DUECARE_PACKAGES:
        raise RuntimeError('Could not install pinned DueCare packages from PyPI or attached wheel datasets.') from exc
    install_source = 'kaggle_wheels'
else:
    # PyPI pip install returned 0 but that does not guarantee `duecare` is
    # importable (a stub package with the same name can satisfy pip while
    # providing none of the real modules). Verify; fall back to wheels if
    # the import is still broken.
    if DUECARE_PACKAGES and not _duecare_importable():
        print('PyPI install finished but duecare.core is not importable. Falling back to Kaggle wheels.')
        if _wheel_install():
            install_source = 'kaggle_wheels'
        else:
            raise RuntimeError('duecare.core not importable after pip and wheel install. Attach taylorsamarel/duecare-llm-wheels and rerun.')

import duecare.core
print(f'DueCare version: {duecare.core.__version__} ({install_source})')
if duecare.core.__version__ != '0.1.0':
    print('Expected DueCare version: 0.1.0')


# 620: DueCare Demo API Endpoint Tour

**The deployment surface without a running server.** This notebook walks every FastAPI endpoint the DueCare demo app (<code>src/demo/app.py</code>) exposes: 17 routes covering single-prompt analysis, batch analysis, full rubric evaluation, native function calling, multimodal document analysis, direct case-example retrieval, multi-document NGO case intake in both JSON and file-upload form, RAG retrieval, quick-check triage, metadata listings, aggregate stats, a liveness probe, the analyst viewer, and the root HTML dashboard. Each endpoint gets a titled subsection with a curl example, a sample response shape, and a Python <code>requests</code> snippet. A Plotly sankey at the bottom shows which endpoints call which downstream agents and tools so the deployment story is visible at a glance.

DueCare is an on-device LLM safety system built on Gemma 4 and named for the common-law duty of care codified in California Civil Code section 1714(a). This tour is the fastest way for a judge or an adopting NGO engineer to understand the demo surface without spinning up uvicorn.

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 6px 10px; text-align: left; width: 22%;">Field</th>
      <th style="padding: 6px 10px; text-align: left; width: 78%;">Value</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px;"><b>Inputs</b></td><td style="padding: 6px 10px;">An in-notebook <code>ENDPOINTS</code> catalog describing the 17 FastAPI routes exposed by <code>src/demo/app.py</code>, plus an optional in-process <code>fastapi.testclient.TestClient</code> bound to the real app so the calls render live output when the package is installed. No external server is required.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Outputs</b></td><td style="padding: 6px 10px;">Per-endpoint subsections with <code>curl</code>, request and response shape, and a Python <code>requests</code> snippet for eight representative routes (<code>/analyze</code>, <code>/rag-context</code>, <code>/function-call</code>, <code>/analyze-document</code>, <code>/migration-case</code>, <code>/migration-case-upload</code>, <code>/evaluate</code>, <code>/quick-check</code>); an HTML summary table covering all 17 endpoints, including a live catalog-vs-app drift audit when the TestClient import succeeds; and a Plotly sankey rendering the endpoint -> agent/tool call graph.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Prerequisites</b></td><td style="padding: 6px 10px;">Kaggle CPU kernel with internet enabled and the <code>taylorsamarel/duecare-llm-wheels</code> wheel dataset attached. The <code>fastapi.testclient</code> path is optional; when the DueCare demo package is not importable the cells print scripted sample responses so the tour still renders end-to-end.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Runtime</b></td><td style="padding: 6px 10px;">Under 30 seconds end-to-end. No model inference; all work is catalog rendering and local TestClient calls.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Pipeline position</b></td><td style="padding: 6px 10px;">Solution Surfaces. Previous: <a href='https://www.kaggle.com/code/taylorsamarel/610-duecare-submission-walkthrough'>610 Submission Walkthrough</a>. Next: <a href='https://www.kaggle.com/code/taylorsamarel/650-duecare-custom-domain-walkthrough'>650 Custom Domain Walkthrough</a>. Section close: <a href='https://www.kaggle.com/code/taylorsamarel/duecare-solution-surfaces-conclusion'>899 Solution Surfaces Conclusion</a>.</td></tr>
  </tbody>
</table>


### Why CPU-only

The tour never loads a model. When the demo package is installed, the notebook uses <code>fastapi.testclient.TestClient</code> to call the real app in-process; when it is not, the scripted example responses still render so the cells always produce output. Either way, no GPU, no network round-trip, no external server.

### Reading order

- **Previous step:** [610 Submission Walkthrough](https://www.kaggle.com/code/taylorsamarel/610-duecare-submission-walkthrough) explains the end-to-end story this API serves.
- **Grading logic upstream:** [140 Evaluation Mechanics](https://www.kaggle.com/code/taylorsamarel/140-duecare-evaluation-mechanics), [260 RAG Comparison](https://www.kaggle.com/code/taylorsamarel/duecare-260-rag-comparison), [460 Citation Verifier](https://www.kaggle.com/code/taylorsamarel/duecare-460-citation-verifier).
- **Next step:** [650 Custom Domain Walkthrough](https://www.kaggle.com/code/taylorsamarel/650-duecare-custom-domain-walkthrough) shows adopters how to plug a new domain pack into the same endpoints.
- **Section close:** [899 Solution Surfaces Conclusion](https://www.kaggle.com/code/taylorsamarel/duecare-solution-surfaces-conclusion).
- **Back to navigation:** [000 Index](https://www.kaggle.com/code/taylorsamarel/duecare-000-index).

### What this notebook does

1. Define a typed <code>ENDPOINTS</code> catalog describing all 17 routes.
2. Optionally spin up a <code>TestClient</code> against <code>duecare.demo.api.app</code>; fall back cleanly when the package is not importable, and compare the live app routes against the catalog when import succeeds.
3. Walk eight representative endpoints (<code>/analyze</code>, <code>/rag-context</code>, <code>/function-call</code>, <code>/analyze-document</code>, <code>/migration-case</code>, <code>/migration-case-upload</code>, <code>/evaluate</code>, <code>/quick-check</code>) with curl + response + Python snippets.
4. Render an HTML summary table covering all 17 endpoints.
5. Render a Plotly sankey of the endpoint -> agent/tool call graph.


---

## At a glance

Walk all 17 FastAPI endpoints with curl examples and live drift audit.


---

## 1. The DueCare API at a glance

The demo app ships 15 routes under the <code>/api/v1</code> prefix plus the analyst viewer and the root HTML dashboard. The <code>ENDPOINTS</code> catalog below captures the method, path, a one-line summary, the request and response shape, a curl example, and a Python <code>requests</code> snippet for each route. Every downstream cell reads from this one source of truth so if a route signature changes, one edit propagates to every subsection.


In [ ]:
import json
ENDPOINTS = json.loads(r'''
[
  {
    "method": "POST",
    "path": "/api/v1/analyze",
    "name": "Single-prompt safety analysis",
    "summary": "Score one input against all loaded trafficking rubrics and return grade, action, indicators, warning text, legal refs, and resources.",
    "request_shape": {
      "text": "str (required)",
      "context": "enum: job_posting | chat | contract | comment | document | other",
      "jurisdiction": "str (optional; corridor code like PH_HK)",
      "language": "enum: en | tl (default en)"
    },
    "response_shape": {
      "score": "float in [0, 1]",
      "grade": "enum: best | good | neutral | bad | worst",
      "action": "enum: pass | warn | review | block",
      "indicators": "list[str]",
      "warning_text": "str",
      "legal_refs": "list[str]",
      "resources": "list[{name, number, url, jurisdiction}]",
      "rubric_results": "list[{rubric_name, score, grade, criteria_results}]"
    },
    "curl_example": "curl -X POST http://localhost:8080/api/v1/analyze \\\n  -H 'Content-Type: application/json' \\\n  -d '{\"text\": \"Agency wants six months of wages as placement fee\", \"context\": \"chat\", \"jurisdiction\": \"PH_HK\"}'",
    "python_example": "import requests\nr = requests.post('http://localhost:8080/api/v1/analyze', json={\n    'text': 'Agency wants six months of wages as placement fee',\n    'context': 'chat',\n    'jurisdiction': 'PH_HK',\n})\nprint(r.json()['grade'], r.json()['action'])"
  },
  {
    "method": "POST",
    "path": "/api/v1/batch",
    "name": "Batch safety analysis",
    "summary": "Accept up to 500 items, score each against the rubric bank, and return per-item results plus aggregate grade and action distributions.",
    "request_shape": {
      "items": "list[AnalyzeRequest] (max 500)"
    },
    "response_shape": {
      "results": "list[AnalyzeResponse]",
      "summary": {
        "total": "int",
        "grade_distribution": "dict[grade, count]",
        "action_distribution": "dict[action, count]",
        "flagged_count": "int"
      },
      "batch_id": "str (uuid-12)"
    },
    "curl_example": "curl -X POST http://localhost:8080/api/v1/batch \\\n  -H 'Content-Type: application/json' \\\n  -d '{\"items\": [{\"text\": \"Agency holds passport until fees paid\"}, {\"text\": \"Live-in caregiver, employer pays all fees per ILO C181\"}]}'",
    "python_example": "import requests\nr = requests.post('http://localhost:8080/api/v1/batch', json={\n    'items': [\n        {'text': 'Agency holds passport until fees paid'},\n        {'text': 'Live-in caregiver, employer pays all fees per ILO C181'},\n    ],\n})\nprint(r.json()['summary'])"
  },
  {
    "method": "POST",
    "path": "/api/v1/evaluate",
    "name": "Full model evaluation",
    "summary": "Send text to Gemma 4 via Ollama and score the response. Modes: plain | rag | guided | compare (all three side-by-side).",
    "request_shape": {
      "text": "str (required)",
      "mode": "enum: plain | rag | guided | compare (default plain)",
      "model": "str (default gemma4:e4b)"
    },
    "response_shape": {
      "plain": "GemmaResult (when mode=compare)",
      "rag": "GemmaResult (when mode=compare)",
      "guided": "GemmaResult (when mode=compare)",
      "response": "str (when mode != compare)",
      "score": "float",
      "grade": "enum",
      "mode_used": "str",
      "latency_ms": "int"
    },
    "curl_example": "curl -X POST http://localhost:8080/api/v1/evaluate \\\n  -H 'Content-Type: application/json' \\\n  -d '{\"text\": \"What should I do about my recruitment fee?\", \"mode\": \"compare\"}'",
    "python_example": "import requests\nr = requests.post('http://localhost:8080/api/v1/evaluate', json={\n    'text': 'What should I do about my recruitment fee?',\n    'mode': 'compare',\n    'model': 'gemma4:e4b',\n})\nprint({k: v.get('grade') for k, v in r.json().items() if isinstance(v, dict)})"
  },
  {
    "method": "POST",
    "path": "/api/v1/function-call",
    "name": "Native Gemma 4 tool call",
    "summary": "Demonstrate Gemma 4 native function calling across the fee, hotline, legal, and exploitation tools.",
    "request_shape": {
      "text": "str (required)"
    },
    "response_shape": {
      "results": "list[{tool, args, result}]",
      "final_answer": "str"
    },
    "curl_example": "curl -X POST http://localhost:8080/api/v1/function-call \\\n  -H 'Content-Type: application/json' \\\n  -d '{\"text\": \"PHP 50000 placement fee for Hong Kong domestic work\"}'",
    "python_example": "import requests\nr = requests.post('http://localhost:8080/api/v1/function-call', json={\n    'text': 'PHP 50000 placement fee for Hong Kong domestic work',\n})\nfor call in r.json()['results']:\n    print(call['tool'])"
  },
  {
    "method": "POST",
    "path": "/api/v1/analyze-document",
    "name": "Multimodal document analysis",
    "summary": "Accept document text or, in production, a photo and extract exploitation indicators, monetary amounts, fee clauses, and passport references.",
    "request_shape": {
      "text": "str (required)",
      "context": "str (optional; default document)"
    },
    "response_shape": {
      "extracted_fields": "dict[str, Any]",
      "indicator_flags": "list[str]",
      "score": "float",
      "grade": "enum",
      "notes": "list[str]"
    },
    "curl_example": "curl -X POST http://localhost:8080/api/v1/analyze-document \\\n  -H 'Content-Type: application/json' \\\n  -d '{\"text\": \"Employer to retain passport for contract duration\", \"context\": \"contract\"}'",
    "python_example": "import requests\nr = requests.post('http://localhost:8080/api/v1/analyze-document', json={\n    'text': 'Employer to retain passport for contract duration',\n    'context': 'contract',\n})\nprint(r.json().get('indicator_flags'))"
  },
  {
    "method": "POST",
    "path": "/api/v1/migration-case",
    "name": "NGO migration case intake",
    "summary": "Bundle multiple migration documents into one case packet, classify each file, build a timeline, retrieve grounding context, and draft complaint text.",
    "request_shape": {
      "case_id": "str (optional)",
      "corridor": "str (optional; corridor code like PH_HK)",
      "documents": "list[{document_id, title, text, context, captured_at}]",
      "include_complaint_templates": "bool (default True)",
      "top_k_context": "int (default 5)"
    },
    "response_shape": {
      "case_id": "str",
      "corridor": "str",
      "document_count": "int",
      "risk_level": "enum: HIGH | MEDIUM | LOW",
      "detected_indicators": "list[str]",
      "applicable_laws": "list[str]",
      "retrieved_context": "str",
      "timeline": "list[{date, label, document_id, description}]",
      "document_analyses": "list[{document_id, title, risk_level, findings, indicator_flags}]",
      "recommended_actions": "list[str]",
      "hotlines": "list[{name, number, url, jurisdiction}]",
      "tool_results": "list[{tool, result}]",
      "complaint_templates": "list[{name, audience, text}]"
    },
    "curl_example": "curl -X POST http://localhost:8080/api/v1/migration-case \\\n  -H 'Content-Type: application/json' \\\n  -d '{\"case_id\": \"case-demo-001\", \"corridor\": \"PH_HK\", \"documents\": [{\"title\": \"Agency receipt\", \"context\": \"receipt\", \"text\": \"Receipt for placement fee: HKD 20000 paid by worker.\"}, {\"title\": \"Employment contract\", \"context\": \"contract\", \"text\": \"Employer will retain passport and deduct fees over 7 months.\"}]}'",
    "python_example": "import requests\nr = requests.post('http://localhost:8080/api/v1/migration-case', json={\n    'case_id': 'case-demo-001',\n    'corridor': 'PH_HK',\n    'documents': [\n        {'title': 'Agency receipt', 'context': 'receipt', 'text': 'Receipt for placement fee: HKD 20000 paid by worker.'},\n        {'title': 'Employment contract', 'context': 'contract', 'text': 'Employer will retain passport and deduct fees over 7 months.'},\n    ],\n})\nprint(r.json()['risk_level'], len(r.json()['timeline']))"
  },
  {
    "method": "POST",
    "path": "/api/v1/migration-case-upload",
    "name": "Upload-first NGO case intake",
    "summary": "Accept a multipart upload of receipts, chats, contracts, and notes; normalize the files into a migration case packet before running the same timeline, legal-context, and complaint-drafting workflow as the JSON endpoint.",
    "request_shape": {
      "files": "list[UploadFile] (required)",
      "corridor": "str (optional; corridor code like PH_HK)",
      "case_id": "str (optional)",
      "case_notes": "str (optional)",
      "include_complaint_templates": "bool (default True)",
      "top_k_context": "int (default 5)",
      "document_contexts_json": "JSON dict[str, str] (optional)",
      "document_dates_json": "JSON dict[str, str] (optional)",
      "document_notes_json": "JSON dict[str, str] (optional)"
    },
    "response_shape": {
      "case_id": "str",
      "corridor": "str",
      "document_count": "int",
      "risk_level": "enum: HIGH | MEDIUM | LOW",
      "detected_indicators": "list[str]",
      "timeline": "list[{date, label, document_id, description}]",
      "document_analyses": "list[{document_id, title, risk_level, findings, indicator_flags}]",
      "recommended_actions": "list[str]",
      "complaint_templates": "list[{name, audience, text}]"
    },
    "curl_example": "curl -X POST http://localhost:8080/api/v1/migration-case-upload \\\n  -F 'corridor=PH_HK' \\\n  -F 'case_id=case-upload-001' \\\n  -F 'files=@agency-receipt.txt;type=text/plain' \\\n  -F 'files=@employment-contract.txt;type=text/plain'",
    "python_example": "import requests\nfiles = [\n    ('files', ('agency-receipt.txt', 'Receipt for placement fee: HKD 20000', 'text/plain')),\n    ('files', ('employment-contract.txt', 'Employer will retain passport during contract period.', 'text/plain')),\n]\ndata = {'corridor': 'PH_HK', 'case_id': 'case-upload-001'}\nr = requests.post('http://localhost:8080/api/v1/migration-case-upload', files=files, data=data)\nprint(r.json()['risk_level'], len(r.json()['timeline']))"
  },
  {
    "method": "POST",
    "path": "/api/v1/rag-context",
    "name": "RAG retrieval for a prompt",
    "summary": "Retrieve legal provisions, corridor fingerprints, and scheme fingerprints from the DueCare knowledge base for prompt injection.",
    "request_shape": {
      "text": "str (required)",
      "top_k": "int (default 5)"
    },
    "response_shape": {
      "query": "str",
      "context": "str (newline-joined, citation-bracketed)",
      "n_entries": "int",
      "n_retrieved": "int"
    },
    "curl_example": "curl -X POST http://localhost:8080/api/v1/rag-context \\\n  -H 'Content-Type: application/json' \\\n  -d '{\"text\": \"Is a placement fee legal for Filipino domestic workers?\", \"top_k\": 5}'",
    "python_example": "import requests\nr = requests.post('http://localhost:8080/api/v1/rag-context', json={\n    'text': 'Is a placement fee legal for Filipino domestic workers?',\n    'top_k': 5,\n})\nprint(r.json()['n_retrieved'], 'entries retrieved')"
  },
  {
    "method": "POST",
    "path": "/api/v1/quick-check",
    "name": "Fast keyword-only check",
    "summary": "Stage 1 enterprise triage. Keyword and regex match under 1 ms. High recall; decides whether to escalate to the full rubric.",
    "request_shape": {
      "text": "str (required)"
    },
    "response_shape": {
      "should_trigger": "bool",
      "score": "float in [0, 1]",
      "matched_keywords": "list[str]",
      "matched_patterns": "list[str]",
      "category_hints": "list[str]"
    },
    "curl_example": "curl -X POST http://localhost:8080/api/v1/quick-check \\\n  -H 'Content-Type: application/json' \\\n  -d '{\"text\": \"Agency holds passport until fees paid\"}'",
    "python_example": "import requests\nr = requests.post('http://localhost:8080/api/v1/quick-check', json={\n    'text': 'Agency holds passport until fees paid',\n})\nprint(r.json()['should_trigger'], r.json()['matched_keywords'])"
  },
  {
    "method": "GET",
    "path": "/api/v1/domains",
    "name": "List available domain packs",
    "summary": "Return metadata for every domain pack loaded in the current process.",
    "request_shape": {},
    "response_shape": {
      "id": "str",
      "display_name": "str",
      "version": "str",
      "description": "str",
      "n_rubrics": "int"
    },
    "curl_example": "curl http://localhost:8080/api/v1/domains",
    "python_example": "import requests\nr = requests.get('http://localhost:8080/api/v1/domains')\nfor d in r.json():\n    print(d['id'])"
  },
  {
    "method": "GET",
    "path": "/api/v1/rubrics",
    "name": "List available rubrics",
    "summary": "Return metadata for every rubric the scorer has loaded.",
    "request_shape": {},
    "response_shape": {
      "name": "str",
      "category": "str",
      "n_criteria": "int",
      "weight": "float"
    },
    "curl_example": "curl http://localhost:8080/api/v1/rubrics",
    "python_example": "import requests\nr = requests.get('http://localhost:8080/api/v1/rubrics')\nprint(len(r.json()), 'rubrics loaded')"
  },
  {
    "method": "GET",
    "path": "/api/v1/case-examples",
    "name": "List bundled case examples",
    "summary": "Return the built-in case-study fixtures that feed the demo viewer and the NGO workflow walkthrough.",
    "request_shape": {},
    "response_shape": {
      "examples": "list[{id, title, corridor, summary, document_count}]"
    },
    "curl_example": "curl http://localhost:8080/api/v1/case-examples",
    "python_example": "import requests\nr = requests.get('http://localhost:8080/api/v1/case-examples')\nprint(len(r.json().get('examples', [])), 'bundled examples')"
  },
  {
    "method": "GET",
    "path": "/api/v1/case-examples/{example_id}",
    "name": "Fetch one bundled case example",
    "summary": "Load a single named demo case with its documents, timeline metadata, and corridor label so an operator can inspect or replay it end to end.",
    "request_shape": {
      "example_id": "path param: str"
    },
    "response_shape": {
      "id": "str",
      "title": "str",
      "corridor": "str",
      "documents": "list[{document_id, title, text, context, captured_at}]"
    },
    "curl_example": "curl http://localhost:8080/api/v1/case-examples/case-demo-001",
    "python_example": "import requests\nr = requests.get('http://localhost:8080/api/v1/case-examples/case-demo-001')\nprint(r.json().get('id'), len(r.json().get('documents', [])))"
  },
  {
    "method": "GET",
    "path": "/api/v1/stats",
    "name": "Aggregate usage stats",
    "summary": "Report totals for analyses since startup: grade distribution, action distribution, top indicators, rubrics loaded, and uptime.",
    "request_shape": {},
    "response_shape": {
      "total_analyses": "int",
      "grade_distribution": "dict[grade, count]",
      "action_distribution": "dict[action, count]",
      "top_indicators": "list[{indicator, count}]",
      "uptime_seconds": "int"
    },
    "curl_example": "curl http://localhost:8080/api/v1/stats",
    "python_example": "import requests\nr = requests.get('http://localhost:8080/api/v1/stats')\nprint(r.json().get('total_analyses'))"
  },
  {
    "method": "GET",
    "path": "/api/v1/health",
    "name": "Health probe",
    "summary": "Cheap liveness endpoint for container orchestration and smoke tests.",
    "request_shape": {},
    "response_shape": {
      "status": "str (literal 'ok')",
      "version": "str",
      "rubrics_loaded": "int",
      "model_id": "str",
      "uptime_seconds": "float"
    },
    "curl_example": "curl http://localhost:8080/api/v1/health",
    "python_example": "import requests\nr = requests.get('http://localhost:8080/api/v1/health')\nprint(r.json()['status'], r.json()['version'])"
  },
  {
    "method": "GET",
    "path": "/viewer",
    "name": "Interactive report viewer",
    "summary": "Serve the richer analyst viewer used for report inspection, HTML exports, and narrative handoff from the dashboard layer.",
    "request_shape": {},
    "response_shape": {
      "Content-Type": "text/html",
      "body": "HTML document (no JSON contract)"
    },
    "curl_example": "curl http://localhost:8080/viewer",
    "python_example": "import requests\nr = requests.get('http://localhost:8080/viewer')\nprint(r.headers['content-type'], len(r.text), 'bytes')"
  },
  {
    "method": "GET",
    "path": "/",
    "name": "HTML dashboard",
    "summary": "Serve the browser dashboard for analysts and demo reviewers.",
    "request_shape": {},
    "response_shape": {
      "Content-Type": "text/html",
      "body": "HTML document (no JSON contract)"
    },
    "curl_example": "curl http://localhost:8080/",
    "python_example": "import requests\nr = requests.get('http://localhost:8080/')\nprint(r.headers['content-type'], len(r.text), 'bytes')"
  }
]
''')


---

## 2. Spin up a local TestClient (optional)

When the DueCare demo package is installed, <code>fastapi.testclient.TestClient</code> imports the FastAPI app directly and calls endpoints in-process, so every subsection below renders live JSON output. When the package is missing on the Kaggle CPU kernel, the cells fall back to scripted sample responses so the tour still reads end-to-end.


In [ ]:
CLIENT_AVAILABLE = False
client = None
client_message = ''

try:
    from fastapi.testclient import TestClient
    try:
        from duecare.demo.api import app  # type: ignore
    except Exception:
        try:
            import sys
            from pathlib import Path as _Path
            repo_root = _Path('.').resolve()
            if (repo_root / 'src' / 'demo' / 'app.py').exists():
                sys.path.insert(0, str(repo_root))
                from src.demo.app import app  # type: ignore
            else:
                raise ImportError('demo app not on path')
        except Exception as inner_exc:
            raise inner_exc
    client = TestClient(app)
    CLIENT_AVAILABLE = bool(client)
    client_message = 'LIVE TestClient bound to the DueCare demo app (in-process, no server needed)'
except Exception as exc:
    CLIENT_AVAILABLE = False
    client = None
    client_message = (
        f'SCRIPTED fallback ({exc.__class__.__name__}: {exc}). '
        'Sample response bodies will render below; install the DueCare demo package '
        'to switch to live TestClient output.'
    )

print(client_message)
print(f'CLIENT_AVAILABLE = {CLIENT_AVAILABLE}')

catalog_paths = sorted(ep['path'] for ep in ENDPOINTS)
print(f'Catalog routes: {len(catalog_paths)}')
if CLIENT_AVAILABLE:
    actual_paths = sorted(
        {
            route.path
            for route in app.routes
            if route.path.startswith('/api/v1/') or route.path in {'/', '/viewer'}
        }
    )
    catalog_only = sorted(set(catalog_paths) - set(actual_paths))
    app_only = sorted(set(actual_paths) - set(catalog_paths))
    print(f'Live app routes: {len(actual_paths)}')
    print('Catalog-only routes:', catalog_only or '(none)')
    print('App-only routes:', app_only or '(none)')
else:
    print('Route drift audit skipped: live app import unavailable.')


---

## 3. POST /api/v1/analyze - one-prompt safety analysis

The primary endpoint behind the video demo. One text in; a grade, an action, indicators, warning text, legal refs, and resource links out. Scored against every rubric loaded by the WeightedRubricScorer.


In [ ]:
import json
ANALYZE_REQUEST = {
    'text': 'My recruitment agency is charging me six months of wages as a placement fee and will hold my passport until it is paid.',
    'context': 'chat',
    'jurisdiction': 'PH_HK',
    'language': 'en',
}
ANALYZE_SAMPLE_RESPONSE = {
    'score': 0.12,
    'grade': 'worst',
    'action': 'block',
    'indicators': ['excessive_placement_fee', 'passport_retention', 'salary_deduction_scheme'],
    'warning_text': 'This arrangement shows multiple trafficking indicators. Do not accept.',
    'legal_refs': ['ILO C181 Article 7 (no worker-paid fees)', 'RA 8042 (PH)', 'Saudi Labor Law Art. 40'],
    'resources': [
        {'name': 'POEA Hotline', 'number': '1343', 'url': None, 'jurisdiction': 'PH'},
        {'name': 'BP2MI', 'number': None, 'url': 'https://bp2mi.go.id', 'jurisdiction': 'ID'},
    ],
    'rubric_results': [],
}

ep = next(e for e in ENDPOINTS if e['path'] == '/api/v1/analyze')
print(f'== {ep["method"]} {ep["path"]} ==')
print()
print('-- curl --')
print(ep['curl_example'])
print()
print('-- Python --')
print(ep['python_example'])
print()
print('-- Response --')
if CLIENT_AVAILABLE:
    try:
        resp = client.post('/api/v1/analyze', json=ANALYZE_REQUEST)
        payload = resp.json()
        print(f'[LIVE TestClient] status={resp.status_code}')
        print(json.dumps({k: payload.get(k) for k in ('score', 'grade', 'action', 'indicators')}, indent=2))
    except Exception as exc:
        print(f'[LIVE call failed: {exc.__class__.__name__}] Scripted sample below:')
        print(json.dumps(ANALYZE_SAMPLE_RESPONSE, indent=2))
else:
    print('[SCRIPTED sample]')
    print(json.dumps(ANALYZE_SAMPLE_RESPONSE, indent=2))


---

## 4. POST /api/v1/rag-context - retrieve legal and corridor context

The retrieval endpoint that feeds the guided-context mode in [260 RAG Comparison](https://www.kaggle.com/code/taylorsamarel/duecare-260-rag-comparison). Given a query, returns up to <code>top_k</code> citation-bracketed entries from the DueCare knowledge base (ILO conventions, corridor fingerprints, scheme fingerprints) for injection into Gemma's prompt.


In [ ]:
import json
RAG_REQUEST = {
    'text': 'Is a placement fee legal for a Filipino domestic worker going to Hong Kong?',
    'top_k': 5,
}
RAG_SAMPLE_RESPONSE = {
    'query': RAG_REQUEST['text'],
    'context': (
        '[1] ILO C181 Article 7: Private employment agencies shall not charge workers... '
        '[2] RA 8042 (PH Migrant Workers Act): Prohibits placement fees beyond one-month salary... '
        '[3] PH_HK corridor fingerprint: standard fee is HKD 0 (employer-paid)... '
        '[4] Scheme fingerprint: six-month salary deduction is a salary-skimming pattern... '
        '[5] POEA Memorandum Circular No. 02 s.2018: no worker-paid placement fees for HK deployment...'
    ),
    'n_entries': 142,
    'n_retrieved': 5,
}

ep = next(e for e in ENDPOINTS if e['path'] == '/api/v1/rag-context')
print(f'== {ep["method"]} {ep["path"]} ==')
print()
print('-- curl --')
print(ep['curl_example'])
print()
print('-- Python --')
print(ep['python_example'])
print()
print('-- Response --')
if CLIENT_AVAILABLE:
    try:
        resp = client.post('/api/v1/rag-context', json=RAG_REQUEST)
        payload = resp.json()
        print(f'[LIVE TestClient] status={resp.status_code}')
        print(json.dumps({k: payload.get(k) for k in ('query', 'n_entries', 'n_retrieved')}, indent=2))
        print('context preview:', (payload.get('context') or '')[:300])
    except Exception as exc:
        print(f'[LIVE call failed: {exc.__class__.__name__}] Scripted sample below:')
        print(json.dumps(RAG_SAMPLE_RESPONSE, indent=2))
else:
    print('[SCRIPTED sample]')
    print(json.dumps(RAG_SAMPLE_RESPONSE, indent=2))


---

## 5. POST /api/v1/function-call - native Gemma 4 tool call

Gemma 4's native function calling exercised against the DueCare tool surface. The server registers five tools (<code>check_fee_legality</code>, <code>check_legal_framework</code>, <code>lookup_hotline</code>, <code>identify_trafficking_indicators</code>, <code>score_exploitation_risk</code>) and routes Gemma's tool choices to their Python implementations. When Ollama is not present, the endpoint still returns a deterministic tool-call trace so the demo pathway stays visible.


In [ ]:
import json
FC_REQUEST = {
    'text': 'The agency charges PHP 50000 placement fee for Hong Kong domestic work.',
}
FC_SAMPLE_RESPONSE = {
    'input': FC_REQUEST['text'],
    'tools_available': [
        'check_fee_legality',
        'check_legal_framework',
        'lookup_hotline',
        'identify_trafficking_indicators',
        'score_exploitation_risk',
    ],
    'tools_called': 5,
    'results': [
        {'tool': 'score_exploitation_risk', 'result': {'risk': 'high', 'score': 0.85}},
        {'tool': 'identify_trafficking_indicators', 'result': {'indicators': ['excessive_fee', 'PH_HK_fee_violation']}},
        {'tool': 'check_fee_legality', 'result': {'country': 'PH', 'legal': False, 'reason': 'Exceeds one-month cap per RA 8042'}},
        {'tool': 'lookup_hotline', 'result': {'country': 'PH', 'number': '1343', 'agency': 'POEA'}},
        {'tool': 'check_legal_framework', 'result': {'jurisdiction': 'PH', 'scenario': 'recruitment_fee', 'statute': 'RA 8042 section 6'}},
    ],
}

ep = next(e for e in ENDPOINTS if e['path'] == '/api/v1/function-call')
print(f'== {ep["method"]} {ep["path"]} ==')
print()
print('-- curl --')
print(ep['curl_example'])
print()
print('-- Python --')
print(ep['python_example'])
print()
print('-- Response --')
if CLIENT_AVAILABLE:
    try:
        resp = client.post('/api/v1/function-call', json=FC_REQUEST)
        payload = resp.json()
        print(f'[LIVE TestClient] status={resp.status_code}')
        print(json.dumps({
            'tools_available': payload.get('tools_available'),
            'tools_called': payload.get('tools_called'),
            'first_result': (payload.get('results') or [{}])[0],
        }, indent=2, default=str))
    except Exception as exc:
        print(f'[LIVE call failed: {exc.__class__.__name__}] Scripted sample below:')
        print(json.dumps(FC_SAMPLE_RESPONSE, indent=2))
else:
    print('[SCRIPTED sample]')
    print(json.dumps(FC_SAMPLE_RESPONSE, indent=2))


---

## 6. POST /api/v1/analyze-document - multimodal document analysis

Evidence surface for the "Gemma 4's unique features are load-bearing" claim. In production the endpoint accepts an image directly through Gemma 4's multimodal path; in this CPU-safe tour it uses already-extracted text as the stand-in and returns the same structured risk, extracted_fields, indicator_flags, and timeline markers payload. OCR is a fallback integration path here, not a product requirement.


In [ ]:
import json
DOC_REQUEST = {
    'text': 'Employment Contract. Employee: Maria R. (composite). Employer to retain passport for the duration of the contract period. Placement fee: HKD 20000, deducted over 7 months.',
    'context': 'contract',
}
DOC_SAMPLE_RESPONSE = {
    'document_type': 'employment_contract',
    'risk_level': 'HIGH',
    'findings': [
        'Document references worker-paid recruitment or placement fees (HKD 20000).',
        'Document describes repayment or deductions taken directly from wages.',
        'Document includes passport or identity-document retention language.',
    ],
    'indicator_flags': ['passport_retention', 'worker_paid_placement_fee', 'salary_deduction_scheme'],
    'legal_refs': ['ILO C181 Art. 7', 'RA 10022', 'Palermo Protocol'],
    'recommendation': 'Pause further payments or contract steps, preserve the document, and escalate it to a labor office or trusted NGO before proceeding.',
    'extracted_fields': {
        'document_type': 'employment_contract',
        'amounts': ['HKD 20000'],
        'dates': [],
        'countries': ['Hong Kong'],
        'corridor_candidates': ['PH_HK'],
    },
    'timeline_markers': [],
    'resources': [
        {'name': 'Immigration Department Help Desk', 'number': '2824 6111', 'jurisdiction': 'HK'},
        {'name': 'IOM Migration Health', 'number': '+41 22 717 9111', 'jurisdiction': 'INTL'},
    ],
    'confidence': 0.73,
}

ep = next(e for e in ENDPOINTS if e['path'] == '/api/v1/analyze-document')
print(f'== {ep["method"]} {ep["path"]} ==')
print()
print('-- curl --')
print(ep['curl_example'])
print()
print('-- Python --')
print(ep['python_example'])
print()
print('-- Response --')
if CLIENT_AVAILABLE:
    try:
        resp = client.post('/api/v1/analyze-document', json=DOC_REQUEST)
        payload = resp.json()
        print(f'[LIVE TestClient] status={resp.status_code}')
        print(json.dumps(payload, indent=2, default=str)[:900])
    except Exception as exc:
        print(f'[LIVE call failed: {exc.__class__.__name__}] Scripted sample below:')
        print(json.dumps(DOC_SAMPLE_RESPONSE, indent=2))
else:
    print('[SCRIPTED sample]')
    print(json.dumps(DOC_SAMPLE_RESPONSE, indent=2))


---

## 7. POST /api/v1/migration-case - NGO case bundle workflow

This is the operator-facing NGO surface the demo was missing before this rebuild. A case worker can bundle receipts, contracts, and recruiter chats from one migration journey; the endpoint then classifies each document, orders the timeline, retrieves legal context, and drafts complaint-ready text without leaving the deployment surface.


In [ ]:
import json
CASE_REQUEST = {
    'case_id': 'case-demo-001',
    'corridor': 'PH_HK',
    'documents': [
        {
            'document_id': 'doc-01',
            'title': 'Agency receipt',
            'context': 'receipt',
            'captured_at': '2026-01-05',
            'text': 'Receipt for placement fee: HKD 20000 paid by worker before deployment.',
        },
        {
            'document_id': 'doc-02',
            'title': 'Employment contract',
            'context': 'contract',
            'captured_at': '2026-01-12',
            'text': 'Employer will retain passport during contract period and deduct fees over 7 months.',
        },
        {
            'document_id': 'doc-03',
            'title': 'Recruiter chat',
            'context': 'chat',
            'captured_at': '2026-01-15',
            'text': 'Pay the remaining fee now or you cannot leave. We will keep your passport until the debt is cleared.',
        },
    ],
}
CASE_SAMPLE_RESPONSE = {
    'case_id': 'case-demo-001',
    'corridor': 'PH_HK',
    'document_count': 3,
    'risk_level': 'HIGH',
    'detected_indicators': ['worker_paid_placement_fee', 'passport_retention', 'debt_bondage_risk'],
    'applicable_laws': ['ILO C181 Art. 7', 'RA 10022', 'Palermo Protocol'],
    'retrieved_context': '[legal_provision] ILO C181 Art. 7 ...',
    'timeline': [
        {'date': '2026-01-05', 'label': 'Payment demand recorded', 'document_id': 'doc-01', 'description': 'Document references worker-paid recruitment or placement fees (HKD 20000).'},
        {'date': '2026-01-12', 'label': 'Contract terms recorded', 'document_id': 'doc-02', 'description': 'Document includes passport or identity-document retention language.'},
        {'date': '2026-01-15', 'label': 'Recruitment conversation captured', 'document_id': 'doc-03', 'description': 'Document creates or acknowledges a debt that could tie the worker to the job.'},
    ],
    'recommended_actions': [
        'Preserve the original files, export metadata, and keep a clean index of what each document shows.',
        'Freeze further worker-paid fees or deductions until a labor-law review confirms they are lawful.',
    ],
    'complaint_templates': [
        {'name': 'ngo_intake_summary', 'audience': 'NGO case worker', 'text': 'Case ID: case-demo-001 ...'},
    ],
}

ep = next(e for e in ENDPOINTS if e['path'] == '/api/v1/migration-case')
print(f'== {ep["method"]} {ep["path"]} ==')
print()
print('-- curl --')
print(ep['curl_example'])
print()
print('-- Python --')
print(ep['python_example'])
print()
print('-- Response --')
if CLIENT_AVAILABLE:
    try:
        resp = client.post('/api/v1/migration-case', json=CASE_REQUEST)
        payload = resp.json()
        print(f'[LIVE TestClient] status={resp.status_code}')
        print(json.dumps(payload, indent=2, default=str)[:1500])
    except Exception as exc:
        print(f'[LIVE call failed: {exc.__class__.__name__}] Scripted sample below:')
        print(json.dumps(CASE_SAMPLE_RESPONSE, indent=2))
else:
    print('[SCRIPTED sample]')
    print(json.dumps(CASE_SAMPLE_RESPONSE, indent=2))


---

## 8. POST /api/v1/migration-case-upload - upload a real case bundle

The JSON case-intake route is what other software calls. This upload route is what an NGO operator or analyst reaches for first: drag in the raw files, attach a corridor label, and let the app normalize them into the same migration-case workflow. It is the notebook surface that proves the demo is usable before a partner writes any integration code.


In [ ]:
import json
UPLOAD_FILES = [
    ('files', ('agency-receipt.txt', 'Receipt for placement fee: HKD 20000 paid by worker before deployment.', 'text/plain')),
    ('files', ('employment-contract.txt', 'Employer will retain passport during contract period and deduct fees over 7 months.', 'text/plain')),
    ('files', ('recruiter-chat.txt', 'Pay the remaining fee now or you cannot leave. We will keep your passport until the debt is cleared.', 'text/plain')),
]
UPLOAD_DATA = {
    'case_id': 'case-upload-001',
    'corridor': 'PH_HK',
    'include_complaint_templates': 'true',
}
UPLOAD_SAMPLE_RESPONSE = {
    'case_id': 'case-upload-001',
    'corridor': 'PH_HK',
    'document_count': 3,
    'risk_level': 'HIGH',
    'detected_indicators': ['worker_paid_placement_fee', 'passport_retention', 'debt_bondage_risk'],
    'timeline': [
        {'date': '2026-01-05', 'label': 'Payment demand recorded', 'document_id': 'agency-receipt.txt', 'description': 'Receipt references worker-paid recruitment or placement fees.'},
        {'date': '2026-01-12', 'label': 'Contract terms recorded', 'document_id': 'employment-contract.txt', 'description': 'Contract includes passport retention language.'},
        {'date': '2026-01-15', 'label': 'Recruitment conversation captured', 'document_id': 'recruiter-chat.txt', 'description': 'Chat escalates debt pressure before deployment.'},
    ],
    'recommended_actions': [
        'Preserve the uploaded originals and export the timeline for the case file.',
        'Escalate the bundle to a labor-rights partner before any further payment or travel step.',
    ],
}

ep = next(e for e in ENDPOINTS if e['path'] == '/api/v1/migration-case-upload')
print(f'== {ep["method"]} {ep["path"]} ==')
print()
print('-- curl --')
print(ep['curl_example'])
print()
print('-- Python --')
print(ep['python_example'])
print()
print('-- Response --')
if CLIENT_AVAILABLE:
    try:
        resp = client.post('/api/v1/migration-case-upload', files=UPLOAD_FILES, data=UPLOAD_DATA)
        payload = resp.json()
        print(f'[LIVE TestClient] status={resp.status_code}')
        print(json.dumps(payload, indent=2, default=str)[:1500])
    except Exception as exc:
        print(f'[LIVE call failed: {exc.__class__.__name__}] Scripted sample below:')
        print(json.dumps(UPLOAD_SAMPLE_RESPONSE, indent=2))
else:
    print('[SCRIPTED sample]')
    print(json.dumps(UPLOAD_SAMPLE_RESPONSE, indent=2))


---

## 9. POST /api/v1/evaluate - full Gemma rubric evaluation

When Ollama is running locally with a Gemma 4 checkpoint pulled, the evaluate endpoint runs the full plain / RAG / guided comparison the video shows, and returns all three modes side by side. On the Kaggle CPU kernel Ollama is not available, so the scripted response below mirrors the schema a live run would emit.


In [ ]:
import json
EVAL_REQUEST = {
    'text': 'What should I do if my recruitment agency is charging me six months of wages?',
    'mode': 'compare',
    'model': 'gemma4:e4b',
}
EVAL_SAMPLE_RESPONSE = {
    'plain':  {'response': 'It depends on the jurisdiction...', 'score': 0.40, 'grade': 'neutral', 'mode_used': 'plain',  'latency_ms': 1280},
    'rag':    {'response': 'ILO C181 prohibits worker-paid fees...', 'score': 0.72, 'grade': 'good',    'mode_used': 'rag',    'latency_ms': 1510},
    'guided': {'response': 'I cannot recommend accepting these terms. ILO C181...', 'score': 0.88, 'grade': 'best', 'mode_used': 'guided', 'latency_ms': 1420},
}

ep = next(e for e in ENDPOINTS if e['path'] == '/api/v1/evaluate')
print(f'== {ep["method"]} {ep["path"]} ==')
print()
print('-- curl --')
print(ep['curl_example'])
print()
print('-- Python --')
print(ep['python_example'])
print()
print('-- Response --')
if CLIENT_AVAILABLE:
    try:
        resp = client.post('/api/v1/evaluate', json=EVAL_REQUEST)
        payload = resp.json()
        print(f'[LIVE TestClient] status={resp.status_code}')
        print(json.dumps(payload, indent=2, default=str)[:900])
    except Exception as exc:
        print(f'[LIVE call failed: {exc.__class__.__name__}] Scripted sample below:')
        print(json.dumps(EVAL_SAMPLE_RESPONSE, indent=2))
else:
    print('[SCRIPTED sample] (live run requires ollama + gemma4:e4b)')
    print(json.dumps(EVAL_SAMPLE_RESPONSE, indent=2))


---

## 10. POST /api/v1/quick-check - fast keyword triage

Stage 1 of the enterprise waterfall. Runs on every message to decide whether to escalate to the full rubric or the Gemma 4 analysis. Latency budget is under 1 ms; high recall is favored over precision so the downstream rubric gets to make the actual call.


In [ ]:
import json
QUICK_REQUEST = {'text': 'Agency holds passport until fees are paid'}
QUICK_SAMPLE_RESPONSE = {
    'should_trigger': True,
    'score': 0.62,
    'matched_keywords': ['passport', 'fees', 'holds'],
    'matched_patterns': ['passport_retention'],
    'category_hints': ['vulnerability_recruitment_violations'],
}

ep = next(e for e in ENDPOINTS if e['path'] == '/api/v1/quick-check')
print(f'== {ep["method"]} {ep["path"]} ==')
print()
print('-- curl --')
print(ep['curl_example'])
print()
print('-- Python --')
print(ep['python_example'])
print()
print('-- Response --')
if CLIENT_AVAILABLE:
    try:
        resp = client.post('/api/v1/quick-check', json=QUICK_REQUEST)
        payload = resp.json()
        print(f'[LIVE TestClient] status={resp.status_code}')
        print(json.dumps(payload, indent=2, default=str))
    except Exception as exc:
        print(f'[LIVE call failed: {exc.__class__.__name__}] Scripted sample below:')
        print(json.dumps(QUICK_SAMPLE_RESPONSE, indent=2))
else:
    print('[SCRIPTED sample]')
    print(json.dumps(QUICK_SAMPLE_RESPONSE, indent=2))


---

## 11. HTML summary table of all 17 endpoints

The eight subsections above covered the representative slice. The table below enumerates every route the demo app exposes so adopters can see the full surface in one place, and prints a live catalog-vs-app drift summary when the TestClient import succeeds.


In [ ]:
from IPython.display import HTML, display

rows_html = []
for ep in ENDPOINTS:
    req_shape = ', '.join(f'<code>{k}</code>' for k in ep['request_shape'].keys()) or '(none)'
    resp_shape = ', '.join(f'<code>{k}</code>' for k in ep['response_shape'].keys())
    rows_html.append(
        '    <tr>'
        f'<td style="padding: 6px 10px;"><code>{ep["method"]}</code></td>'
        f'<td style="padding: 6px 10px;"><code>{ep["path"]}</code></td>'
        f'<td style="padding: 6px 10px;">{ep["summary"]}</td>'
        f'<td style="padding: 6px 10px;">{req_shape}</td>'
        f'<td style="padding: 6px 10px;">{resp_shape}</td>'
        '</tr>'
    )

table_html = (
    '<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">\n'
    '  <thead>\n'
    '    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">\n'
    '      <th style="padding: 6px 10px; text-align: left; width: 8%;">Method</th>\n'
    '      <th style="padding: 6px 10px; text-align: left; width: 22%;">Path</th>\n'
    '      <th style="padding: 6px 10px; text-align: left; width: 35%;">Summary</th>\n'
    '      <th style="padding: 6px 10px; text-align: left; width: 17%;">Request shape</th>\n'
    '      <th style="padding: 6px 10px; text-align: left; width: 18%;">Response shape</th>\n'
    '    </tr>\n'
    '  </thead>\n'
    '  <tbody>\n'
    + '\n'.join(rows_html)
    + '\n  </tbody>\n'
    '</table>'
)
display(HTML(table_html))

catalog_paths = sorted(ep['path'] for ep in ENDPOINTS)
print(f'Catalog routes listed here: {len(catalog_paths)}')
if CLIENT_AVAILABLE:
    actual_paths = sorted(
        {
            route.path
            for route in app.routes
            if route.path.startswith('/api/v1/') or route.path in {'/', '/viewer'}
        }
    )
    print(f'Live app routes imported: {len(actual_paths)}')
    print('Catalog-only routes:', sorted(set(catalog_paths) - set(actual_paths)) or '(none)')
    print('App-only routes:', sorted(set(actual_paths) - set(catalog_paths)) or '(none)')
else:
    print('Route drift audit skipped: live app import unavailable.')


---

## 12. Call-graph diagram (endpoint -> agent / tool)

The sankey below renders which endpoints flow into which downstream agents and tools. It is the visual restatement of the table above: every request path resolves to a concrete handler whose dependencies are named so a reader can trace a deployment feature end-to-end without opening the source.


In [ ]:
import plotly.graph_objects as go

# Edges: source endpoint -> downstream agent / tool. One edge per
# canonical call. Weights are illustrative (higher = hotter path in
# the demo flow).
EDGES = [
    ('/api/v1/analyze',          'WeightedRubricScorer',    8),
    ('/api/v1/analyze',          'AnalysisLog',             4),
    ('/api/v1/batch',            'WeightedRubricScorer',    6),
    ('/api/v1/batch',            'BatchSummaryBuilder',     4),
    ('/api/v1/evaluate',         'Judge agent',             7),
    ('/api/v1/evaluate',         'Scorer agent',            5),
    ('/api/v1/evaluate',         'Ollama / Gemma 4',        6),
    ('/api/v1/function-call',    'Coordinator agent',       8),
    ('/api/v1/function-call',    'Tool: score_exploitation_risk',      4),
    ('/api/v1/function-call',    'Tool: identify_trafficking_indicators', 4),
    ('/api/v1/function-call',    'Tool: check_fee_legality',           3),
    ('/api/v1/function-call',    'Tool: lookup_hotline',               3),
    ('/api/v1/function-call',    'Tool: check_legal_framework',        3),
    ('/api/v1/analyze-document', 'DocumentAnalyzer',        6),
    ('/api/v1/analyze-document', 'Gemma 4 multimodal',      5),
    ('/api/v1/migration-case',   'Case orchestrator',       8),
    ('/api/v1/migration-case',   'DocumentAnalyzer',        6),
    ('/api/v1/migration-case',   'RAGStore',                5),
    ('/api/v1/migration-case',   'Tool: score_exploitation_risk',      4),
    ('/api/v1/migration-case',   'Tool: check_legal_framework',        3),
    ('/api/v1/migration-case',   'Tool: lookup_hotline',               3),
    ('/api/v1/migration-case-upload', 'Upload parser',      5),
    ('/api/v1/migration-case-upload', 'Case orchestrator',  6),
    ('/api/v1/migration-case-upload', 'DocumentAnalyzer',   4),
    ('/api/v1/rag-context',      'RAGStore',                6),
    ('/api/v1/rag-context',      'Retriever',               5),
    ('/api/v1/quick-check',      'QuickFilter',             6),
    ('/api/v1/domains',          'DomainPack registry',     3),
    ('/api/v1/rubrics',          'WeightedRubricScorer',    3),
    ('/api/v1/case-examples',    'CaseExample registry',    3),
    ('/api/v1/case-examples/{example_id}', 'CaseExample registry', 3),
    ('/api/v1/stats',            'AnalysisLog',             3),
    ('/api/v1/health',           'Lifespan state',          2),
    ('/viewer',                  'HTML renderer',           2),
    ('/',                        'HTML renderer',           3),
]

nodes = []
node_index = {}
for src_node, dst_node, _ in EDGES:
    for n in (src_node, dst_node):
        if n not in node_index:
            node_index[n] = len(nodes)
            nodes.append(n)

src_idx = [node_index[s] for s, _, _ in EDGES]
dst_idx = [node_index[d] for _, d, _ in EDGES]
values  = [v for _, _, v in EDGES]

# Color endpoints (left column) blue; downstream nodes (right column)
# green or orange depending on whether they are agents (model-backed)
# or tools / stores (deterministic).
AGENT_NODES = {'Judge agent', 'Scorer agent', 'Coordinator agent', 'Case orchestrator', 'DocumentAnalyzer', 'Ollama / Gemma 4', 'Gemma 4 multimodal'}
def _node_color(name: str) -> str:
    if name.startswith('/'):
        return '#3b82f6'
    if name in AGENT_NODES:
        return '#10b981'
    return '#f97316'

node_colors = [_node_color(n) for n in nodes]

fig = go.Figure(go.Sankey(
    arrangement='snap',
    node=dict(
        label=nodes,
        color=node_colors,
        pad=14,
        thickness=16,
        line=dict(color='#1f2937', width=0.5),
    ),
    link=dict(source=src_idx, target=dst_idx, value=values),
))
fig.update_layout(
    title=dict(text='DueCare API call graph: endpoints -> agents and tools', font_size=16),
    template='plotly_white',
    height=640,
    width=980,
    margin=dict(t=60, b=40, l=20, r=20),
)
fig.show()


---

## What just happened

- Catalogued all 17 FastAPI endpoints the DueCare demo app exposes, with method, path, summary, request shape, response shape, curl example, and Python <code>requests</code> snippet captured in a typed <code>ENDPOINTS</code> list.
- Bound an in-process <code>fastapi.testclient.TestClient</code> when possible so eight representative endpoints returned live JSON; fell back to scripted responses otherwise so the tour always renders.
- Walked <code>/api/v1/analyze</code>, <code>/api/v1/rag-context</code>, <code>/api/v1/function-call</code>, <code>/api/v1/analyze-document</code>, <code>/api/v1/migration-case</code>, <code>/api/v1/migration-case-upload</code>, <code>/api/v1/evaluate</code>, <code>/api/v1/quick-check</code> with full curl + Python + response output.
- Rendered an HTML summary table covering the full 17-endpoint surface and printed a live catalog-vs-app drift audit when the demo app imported successfully.
- Rendered a Plotly sankey of the endpoint -> agent / tool call graph so the deployment story is visible at a glance.

### Key findings

1. **Every deployment feature the video names resolves to one of these 17 routes.** No hidden surface; no private endpoints; nothing held back for the demo.
2. **The eight representative endpoints exercise both Gemma 4's unique features** (function-call for native tool calling, analyze-document for multimodal, migration-case plus migration-case-upload for case-level orchestration) and the supporting surfaces (analyze for rubric, rag-context for retrieval, evaluate for the three-mode comparison, quick-check for waterfall stage 1).
3. **The TestClient path is the safest demo mode.** No uvicorn, no port, no firewall issues; the same app object the server mounts is the one the tests drive, so behavior stays identical between the notebook and production.
4. **The route drift audit keeps the catalog honest.** When the live app imports, the notebook prints catalog-only and app-only routes so a stale endpoint list shows up immediately.
5. **The sankey makes the orchestration story legible in one chart.** /function-call flows into the Coordinator agent, /migration-case and /migration-case-upload fan into the Case orchestrator plus retrieval and hotline/legal tools, /evaluate flows into Judge + Scorer agents that share the Ollama backend, and /rag-context feeds the Retriever that every guided mode depends on.

---

## Troubleshooting

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 6px 10px; text-align: left; width: 38%;">Symptom</th>
      <th style="padding: 6px 10px; text-align: left; width: 62%;">Resolution</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px;">Demo module not importable (<code>CLIENT_AVAILABLE = False</code>).</td><td style="padding: 6px 10px;">Expected on the Kaggle CPU kernel when <code>duecare-llm</code> is not installed. The notebook falls back to scripted sample responses automatically; install the <code>duecare-llm</code> meta package or attach the <code>taylorsamarel/duecare-llm-wheels</code> wheels dataset to switch to live TestClient output.</td></tr>
    <tr><td style="padding: 6px 10px;">An endpoint returns <code>500</code> from the TestClient.</td><td style="padding: 6px 10px;">Inspect the <code>resp.text</code> body for the server-side traceback. The most common cause is a missing rubric config (<code>configs/duecare/domains/trafficking/rubrics/</code>) or an unattached knowledge base for <code>/api/v1/rag-context</code>. Fix upstream and rerun the cell.</td></tr>
    <tr><td style="padding: 6px 10px;">A curl example hangs or times out.</td><td style="padding: 6px 10px;">The curl snippets assume <code>http://localhost:8080</code>. If you are running the app on another host or port, edit the URL; if no server is running, the TestClient cells are the intended path.</td></tr>
    <tr><td style="padding: 6px 10px;"><code>TestClient</code> raises on import.</td><td style="padding: 6px 10px;">Requires <code>fastapi &gt;= 0.100</code> and <code>httpx</code>. Both come with the <code>duecare-llm</code> meta package; if you installed only <code>duecare-llm-core</code> you will need to <code>pip install httpx</code> as well.</td></tr>
    <tr><td style="padding: 6px 10px;">Response shape does not match the table.</td><td style="padding: 6px 10px;">Pydantic models evolve between releases. Check the route drift audit printed under the summary table first; if it shows app-only routes or catalog-only routes, update the <code>ENDPOINTS</code> catalog in this builder before trusting the prose. Then cross-check against <code>src/demo/models.py</code> and open an issue if a field disappeared or renamed without a deprecation.</td></tr>
  </tbody>
</table>


---

## Next

- **Continue the section:** [650 Custom Domain Walkthrough](https://www.kaggle.com/code/taylorsamarel/650-duecare-custom-domain-walkthrough) shows adopters how to plug a new domain pack into these same endpoints.
- **Section close:** [899 Solution Surfaces Conclusion](https://www.kaggle.com/code/taylorsamarel/duecare-solution-surfaces-conclusion).
- **Prior step:** [610 Submission Walkthrough](https://www.kaggle.com/code/taylorsamarel/610-duecare-submission-walkthrough).
- **Dashboards the API feeds:** [600 Results Dashboard](https://www.kaggle.com/code/taylorsamarel/600-duecare-results-dashboard).
- **Back to navigation (optional):** [000 Index](https://www.kaggle.com/code/taylorsamarel/duecare-000-index).


In [ ]:
print(
    'API tour handoff >>> Continue to 650 Custom Domain Walkthrough: '
    'https://www.kaggle.com/code/taylorsamarel/650-duecare-custom-domain-walkthrough'
    '. Section close: 899 Solution Surfaces Conclusion: '
    'https://www.kaggle.com/code/taylorsamarel/duecare-solution-surfaces-conclusion'
    '.'
)
